In [ ]:
import numpy as np
import gradio as gr
import time
import plotly.graph_objects as go
import plotly.express as px
import math
import random
import copy
import pandas as pd

# ==========================================
# 1. БИТОВЫЕ УТИЛИТЫ (Единый источник истины)
# ==========================================
class BitboardUtils:
    def __init__(self):
        self.lines = []
        for r in range(7):
            for c in range(4): self.lines.append(sum(1 << (r * 7 + c + i) for i in range(4)))
        for c in range(7):
            for r in range(4): self.lines.append(sum(1 << ((r + i) * 7 + c) for i in range(4)))
        for r in range(4):
            for c in range(4): self.lines.append(sum(1 << ((r + i) * 7 + (c + i)) for i in range(4)))
        for r in range(4):
            for c in range(3, 7): self.lines.append(sum(1 << ((r + i) * 7 + (c - i)) for i in range(4)))
                
    def check_win_and_line(self, board_int):
        for line in self.lines:
            if (board_int & line) == line:
                return True, line
        return False, 0

    def get_winning_cells_int(self, board_int, opponent_int):
        threats = []
        for i in range(49):
            if not (board_int & (1 << i)) and not (opponent_int & (1 << i)):
                test_board = board_int | (1 << i)
                if self.check_win_and_line(test_board)[0]:
                    threats.append(i)
        return threats

    def get_all_empty(self, my_board, opp_board):
        filled = my_board | opp_board
        return [i for i in range(49) if not (filled & (1 << i))]

bit_utils = BitboardUtils()

# ==========================================
# 2. АГЕНТЫ
# ==========================================
class BitboardMinimaxAgent:
    def __init__(self, name, depth=6, time_limit=3.0):
        self.name = name
        self.max_depth = depth
        self.time_limit = time_limit
        self.stats = {"positions_evaluated": 0, "max_depth_reached": 0}

    def _count_open_threes(self, board, opponent_board):
        open_threes = 0
        for line in bit_utils.lines:
            my_count = bin(board & line).count('1')
            opp_count = bin(opponent_board & line).count('1')
            if my_count == 3 and opp_count == 0:
                empty_idx = None
                for i in range(4):
                    bit_pos = line & (1 << i)
                    if not (board & bit_pos) and not (opponent_board & bit_pos):
                        empty_idx = i
                        break
                if empty_idx is not None:
                    if empty_idx > 0 and empty_idx < 3:
                        open_threes += 1
                    elif empty_idx == 0:
                        first_bit = line & -line
                        first_pos = int(math.log2(first_bit))
                        if first_pos > 0 and not (opponent_board & (1 << (first_pos - 1))):
                            open_threes += 1
                    elif empty_idx == 3:
                        last_bit = 1 << (line.bit_length() - 1)
                        last_pos = int(math.log2(last_bit))
                        if last_pos < 48 and not (opponent_board & (1 << (last_pos + 1))):
                            open_threes += 1
        return open_threes

    def _evaluate(self, my_board, opp_board):
        self.stats["positions_evaluated"] += 1
        score, w = 0, self.weights
        for line in bit_utils.lines:
            my_count = bin(my_board & line).count('1')
            opp_count = bin(opp_board & line).count('1')
            if my_count > 0 and opp_count > 0: continue
            if my_count == 4: score += 100000
            elif my_count == 3: score += w['w3']
            elif my_count == 2: score += w['w2']
            if opp_count == 4: score -= 100000
            elif opp_count == 3: score -= w['block_penalty']
            elif opp_count == 2: score -= w['w2']
        
        # ИЗМЕНЕНО: убраны множители 1.5 и 1.4
        for line in bit_utils.lines:
            if bin(my_board & line).count('1') == 3 and bin(opp_board & line).count('1') == 0: 
                score += w['w_open_three']
            if bin(opp_board & line).count('1') == 3 and bin(my_board & line).count('1') == 0: 
                score -= w['w_open_three']  # Было: * 1.5
        
        my_forks = len(bit_utils.get_winning_cells_int(my_board, opp_board))
        opp_forks = len(bit_utils.get_winning_cells_int(opp_board, my_board))
        if my_forks >= 2: score += w['w_fork']
        if opp_forks >= 2: score -= w['w_fork']  # Было: * 1.4
        
        for idx in [24, 17, 18, 19, 23, 25, 29, 30, 31]:
            if my_board & (1 << idx): score += w['w_center']
            elif opp_board & (1 << idx): score -= w['w_center']
        return score

    def _get_candidates(self, my_board, opp_board):
        filled = my_board | opp_board
        if filled == 0: return [24]
        candidates = []
        for i in range(49):
            if not (filled & (1 << i)):
                r, c = divmod(i, 7)
                is_near = False
                for dr in range(-2, 3):
                    for dc in range(-2, 3):
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < 7 and 0 <= nc < 7:
                            if filled & (1 << (nr * 7 + nc)):
                                is_near = True
                                break
                    if is_near: break
                if is_near: candidates.append(i)
        candidates.sort(key=lambda idx: - (abs(idx // 7 - 3) + abs(idx % 7 - 3)))
        return candidates[:40]

    def _minimax(self, my_board, opp_board, depth, alpha, beta, is_maximizing, start_time):
        if time.time() - start_time > self.time_limit:
            return self._evaluate(my_board, opp_board)
        win, _ = bit_utils.check_win_and_line(my_board)
        if win: return 100000 + depth
        win_opp, _ = bit_utils.check_win_and_line(opp_board)
        if win_opp: return -100000 - depth
        if depth == 0 or (my_board | opp_board) == (1 << 49) - 1:
            return self._evaluate(my_board, opp_board)

        self.stats["max_depth_reached"] = max(self.stats["max_depth_reached"], depth)
        candidates = self._get_candidates(my_board, opp_board)
        if is_maximizing:
            max_eval = -float('inf')
            for move in candidates:
                eval_score = self._minimax(my_board | (1 << move), opp_board, depth - 1, alpha, beta, False, start_time)
                max_eval = max(max_eval, eval_score)
                alpha = max(alpha, eval_score)
                if beta <= alpha: break
            return max_eval
        else:
            min_eval = float('inf')
            for move in candidates:
                eval_score = self._minimax(my_board, opp_board | (1 << move), depth - 1, alpha, beta, True, start_time)
                min_eval = min(min_eval, eval_score)
                beta = min(beta, eval_score)
                if beta <= alpha: break
            return min_eval

    def get_move(self, my_board, opp_board):
        self.stats = {"positions_evaluated": 0}
        wins = bit_utils.get_winning_cells_int(my_board, opp_board)
        if wins: return wins[0]
        blocks = bit_utils.get_winning_cells_int(opp_board, my_board)
        if blocks: return blocks[0]
        
        # НОВОЕ: с вероятностью temperature делаем случайный ход
        candidates = self._get_candidates(my_board, opp_board)
        if random.random() < self.temperature and candidates:
            return random.choice(candidates)
        
        best_move, best_score = candidates[0], -float('inf')
        start_time = time.time()
        for move in candidates:
            if time.time() - start_time > self.time_limit * 0.8: break
            s = self._minimax(my_board | (1 << move), opp_board, self.depth-1, 
                            -float('inf'), float('inf'), False, start_time)
            if s > best_score: best_score, best_move = s, move
        return best_move
    
    def find_best_move_and_eval(self, my_board, opp_board):
        self.stats = {"positions_evaluated": 0, "max_depth_reached": 0}
        wins = bit_utils.get_winning_cells_int(my_board, opp_board)
        if wins: return wins[0], 100000
        blocks = bit_utils.get_winning_cells_int(opp_board, my_board)
        if blocks: return blocks[0], 0
        
        best_move = self._get_candidates(my_board, opp_board)[0]
        best_score = -float('inf')
        start_time = time.time()
        for depth in range(2, self.max_depth + 1, 2):
            if time.time() - start_time > self.time_limit * 0.8: break
            current_best = best_move
            current_score = -float('inf')
            for move in self._get_candidates(my_board, opp_board):
                score = self._minimax(my_board | (1 << move), opp_board, depth - 1, -float('inf'), float('inf'), False, start_time)
                if score > current_score:
                    current_score = score
                    current_best = move
            best_move = current_best
            best_score = current_score
        return best_move, best_score

class MCTSAgent:
    def __init__(self, name="MCTS", simulations=1000, time_limit=3.0):
        self.name = name
        self.simulations = simulations
        self.time_limit = time_limit
        self.stats = {"positions_evaluated": 0, "max_depth_reached": 0}
        
    class Node:
        def __init__(self, my_board, opp_board, parent=None, move=None):
            self.my_board = my_board
            self.opp_board = opp_board
            self.parent = parent
            self.move = move
            self.children = []
            self.wins = 0
            self.visits = 0
            self.untried_moves = bit_utils.get_all_empty(my_board, opp_board)
            
        def ucb1(self, exploration=1.41):
            if self.visits == 0: return float('inf')
            return (self.wins / self.visits) + exploration * math.sqrt(math.log(self.parent.visits) / self.visits)

    def _evaluate(self, my_board, opp_board):
        self.stats["positions_evaluated"] += 1
        score = 0
        for line in bit_utils.lines:
            my_count = bin(my_board & line).count('1')
            opp_count = bin(opp_board & line).count('1')
            if my_count > 0 and opp_count > 0: continue
            if my_count == 4: score += 100000
            elif my_count == 3: score += 5000
            elif my_count == 2: score += 200
            if opp_count == 4: score -= 100000
            elif opp_count == 3: score -= 15000
            elif opp_count == 2: score -= 200
        return score

    def _rollout(self, my_board, opp_board, start_time):
        self.stats["positions_evaluated"] += 1
        current_my, current_opp = my_board, opp_board
        is_maximizing = True
        while True:
            if time.time() - start_time > self.time_limit: return 0
            win, _ = bit_utils.check_win_and_line(current_my)
            if win: return 1 if is_maximizing else -1
            win_opp, _ = bit_utils.check_win_and_line(current_opp)
            if win_opp: return -1 if is_maximizing else 1
            empty = bit_utils.get_all_empty(current_my, current_opp)
            if not empty: return 0
            move = random.choice(empty)
            if is_maximizing: current_my |= (1 << move)
            else: current_opp |= (1 << move)
            is_maximizing = not is_maximizing

    def get_move(self, my_board, opp_board):
        self.stats = {"positions_evaluated": 0, "max_depth_reached": 0}
        wins = bit_utils.get_winning_cells_int(my_board, opp_board)
        if wins: return wins[0]
        blocks = bit_utils.get_winning_cells_int(opp_board, my_board)
        if blocks: return blocks[0]
        
        root = self.Node(my_board, opp_board)
        start_time = time.time()
        for _ in range(self.simulations):
            if time.time() - start_time > self.time_limit: break
            node = root
            while node.untried_moves == [] and node.children:
                node = max(node.children, key=lambda n: n.ucb1())
            if node.untried_moves:
                move = random.choice(node.untried_moves)
                node.untried_moves.remove(move)
                new_my = my_board | (1 << move) if node == root else node.opp_board
                new_opp = opp_board if node == root else node.my_board | (1 << move)
                child = self.Node(new_my, new_opp, parent=node, move=move)
                node.children.append(child)
                node = child
            result = self._rollout(node.my_board, node.opp_board, start_time)
            while node:
                node.visits += 1
                if (node.parent and node.parent == root and result > 0) or (not node.parent and result > 0):
                    node.wins += 1
                elif result == 0:
                    node.wins += 0.5
                node = node.parent
        if not root.children: return bit_utils.get_all_empty(my_board, opp_board)[0]
        return max(root.children, key=lambda c: c.visits).move
    
    def find_best_move_and_eval(self, my_board, opp_board):
        return self.get_move(my_board, opp_board), self._evaluate(my_board, opp_board)

class ClassicTFBAgent:
    def __init__(self):
        self.np_lines = [np.array([i for i in range(49) if (line & (1 << i))]) for line in bit_utils.lines]
        self.stats = {"positions_evaluated": 0, "max_depth_reached": 1}

    def _get_winning_cells(self, board, player):
        winning_cells = []
        for line in self.np_lines:
            cells = board[line]
            if np.sum(cells == player) == 3 and np.sum(cells == 0) == 1:
                winning_cells.append(line[np.where(cells == 0)[0][0]])
        return winning_cells

    def get_move(self, board, player):
        self.stats = {"positions_evaluated": 0, "max_depth_reached": 1}
        opp = -player
        my_wins = self._get_winning_cells(board, player)
        if my_wins: return my_wins[0]
        opp_wins = self._get_winning_cells(board, opp)
        if opp_wins: return opp_wins[0]
        valid = np.where(board == 0)[0]
        best_move, best_score = valid[0], -float('inf')
        for move in valid[:25]:
            self.stats["positions_evaluated"] += 1
            board[move] = player
            score = len(self._get_winning_cells(board, player)) * 100 - len(self._get_winning_cells(board, opp)) * 150
            board[move] = 0
            if score > best_score:
                best_score, best_move = score, move
        return best_move
    
    def _evaluate(self, my_board, opp_board): return 0
    def find_best_move_and_eval(self, my_board, opp_board): return self.get_move(my_board, 1), 0

class EvolvableAgent:
    PARAM_NAMES = ['w2', 'w3', 'w_open_three', 'w_fork', 'w_center', 'block_penalty']
    
    def __init__(self, weights=None, depth=3, time_limit=0.05, temperature=0.25):
        self.depth = depth
        self.time_limit = time_limit
        self.temperature = temperature # 25% шанс на полный хаос
        self.stats = {"positions_evaluated": 0}
        self.weights = weights.copy() if weights else {
            'w2': 50,           # Двойки почти не важны
            'w3': 15000,        # АТАКА: Строить свои тройки очень выгодно
            'w_open_three': 8000, # ЗАЩИТА: Снижена. Лучше атаковать, чем паниковать
            'w_fork': 12000,    # ЗАЩИТА/АТАКА: Умеренная ценность
            'w_center': 100,    # Контроль центра важен
            'block_penalty': 6000, # ЗАЩИТА: Меньше, чем награда за свою атаку (15000)!
        }
    
    def _evaluate(self, my_board, opp_board):
        self.stats["positions_evaluated"] += 1
        score, w = 0, self.weights
        for line in bit_utils.lines:
            my_count = bin(my_board & line).count('1')
            opp_count = bin(opp_board & line).count('1')
            if my_count > 0 and opp_count > 0: continue
            if my_count == 4: score += 100000
            elif my_count == 3: score += w['w3']
            elif my_count == 2: score += w['w2']
            if opp_count == 4: score -= 100000
            elif opp_count == 3: score -= w['block_penalty']
            elif opp_count == 2: score -= w['w2']
        
        # Упрощенная оценка триплетов и вилок без раздувающих множителей
        for line in bit_utils.lines:
            if bin(my_board & line).count('1') == 3 and bin(opp_board & line).count('1') == 0: 
                score += w['w_open_three']
            if bin(opp_board & line).count('1') == 3 and bin(my_board & line).count('1') == 0: 
                score -= w['w_open_three']
        
        my_forks = len(bit_utils.get_winning_cells_int(my_board, opp_board))
        opp_forks = len(bit_utils.get_winning_cells_int(opp_board, my_board))
        if my_forks >= 2: score += w['w_fork']
        if opp_forks >= 2: score -= w['w_fork']
        
        for idx in [24, 17, 18, 19, 23, 25, 29, 30, 31]:
            if my_board & (1 << idx): score += w['w_center']
            elif opp_board & (1 << idx): score -= w['w_center']
        return score
    
    def _get_candidates(self, my_board, opp_board):
        filled = my_board | opp_board
        if filled == 0: return [24]
        candidates = []
        for i in range(49):
            if not (filled & (1 << i)):
                r, c = divmod(i, 7)
                is_near = any(filled & (1 << (nr * 7 + nc)) for dr in range(-2, 3) for dc in range(-2, 3) if 0 <= (nr:=r+dr) < 7 and 0 <= (nc:=c+dc) < 7)
                if is_near: candidates.append(i)
        candidates.sort(key=lambda idx: -(abs(idx // 7 - 3) + abs(idx % 7 - 3)))
        return candidates[:30]
    
    def _minimax(self, my_board, opp_board, depth, alpha, beta, is_max, start_time):
        if time.time() - start_time > self.time_limit: return self._evaluate(my_board, opp_board)
        if bit_utils.check_win_and_line(my_board)[0]: return 100000 + depth
        if bit_utils.check_win_and_line(opp_board)[0]: return -100000 - depth
        if depth == 0: return self._evaluate(my_board, opp_board)
        
        candidates = self._get_candidates(my_board, opp_board)
        if is_max:
            max_eval = -float('inf')
            for move in candidates:
                s = self._minimax(my_board | (1 << move), opp_board, depth-1, alpha, beta, False, start_time)
                max_eval, alpha = max(max_eval, s), max(alpha, s)
                if beta <= alpha: break
            return max_eval
        else:
            min_eval = float('inf')
            for move in candidates:
                s = self._minimax(my_board, opp_board | (1 << move), depth-1, alpha, beta, True, start_time)
                min_eval, beta = min(min_eval, s), min(beta, s)
                if beta <= alpha: break
            return min_eval
    
    def get_move(self, my_board, opp_board):
        self.stats = {"positions_evaluated": 0}
        
        # 1. Мгновенная победа (всегда приоритет)
        wins = bit_utils.get_winning_cells_int(my_board, opp_board)
        if wins: return wins[0]
        
        # 2. РЕЖИМ ХАОСА: 25% шанс сделать АБСОЛЮТНО случайный ход из ВСЕХ пустых клеток
        # Это ломает симметрию и создает фатальные ошибки для эволюции
        if random.random() < self.temperature:
            all_empty = bit_utils.get_all_empty(my_board, opp_board)
            if all_empty:
                return random.choice(all_empty)
        
        # 3. Мгновенная блокировка (только если не сработал хаос)
        blocks = bit_utils.get_winning_cells_int(opp_board, my_board)
        if blocks: return blocks[0]
        
        # 4. Обычный поиск
        candidates = self._get_candidates(my_board, opp_board)
        if not candidates:
            all_empty = bit_utils.get_all_empty(my_board, opp_board)
            return all_empty[0] if all_empty else -1
            
        best_move, best_score = candidates[0], -float('inf')
        start_time = time.time()
        for move in candidates:
            if time.time() - start_time > self.time_limit * 0.8: break
            s = self._minimax(my_board | (1 << move), opp_board, self.depth-1, -float('inf'), float('inf'), False, start_time)
            if s > best_score: best_score, best_move = s, move
        return best_move
    
    def crossover(self, other):
        child_weights = {k: self.weights[k] * random.random() + other.weights[k] * (1 - random.random()) for k in self.weights}
        return EvolvableAgent(weights=child_weights, depth=self.depth, time_limit=self.time_limit, temperature=self.temperature)
    
    def mutate(self, mutation_rate=0.4, mutation_scale=0.4): # Увеличили мутацию для разнообразия
        for key in self.weights:
            if random.random() < mutation_rate:
                self.weights[key] = max(1, self.weights[key] + self.weights[key] * mutation_scale * (random.random() * 2 - 1))
    
    def clone(self):
        return EvolvableAgent(weights=self.weights.copy(), depth=self.depth, time_limit=self.time_limit, temperature=self.temperature)

# Глобальные агенты для GUI
agent_absolute = BitboardMinimaxAgent(name="Absolute", depth=6, time_limit=3.0)
agent_grandmaster = BitboardMinimaxAgent(name="Grandmaster", depth=8, time_limit=10.0)
agent_strong = BitboardMinimaxAgent(name="Strong Bitboard", depth=4, time_limit=1.5)
agent_mcts = MCTSAgent(name="MCTS", simulations=1000, time_limit=3.0)
agent_classic = ClassicTFBAgent()

champion_weights = {
    'w2': 32.66, 
    'w3': 11170.91, 
    'w_open_three': 11876.38,
    'w_fork': 2576.53, 
    'w_center': 23.90, 
    'block_penalty': 6387.58
}

# Эволюционный чемпион - обучен играть против себя
agent_champion = EvolvableAgent(
    weights=champion_weights, 
    depth=4, 
    time_limit=2.0, 
    temperature=0.0  # В реальной игре без случайности!
)

# ==========================================
# 3. ОЦЕНКА КАЧЕСТВА ХОДОВ
# ==========================================
def evaluate_move_quality(actual_move, actual_eval, best_move, best_eval, mode="win"):
    if actual_move == best_move: return "🌟"
    eval_diff = best_eval - actual_eval
    thresholds = (500, 2000, 5000) if mode == "win" else (1000, 3000, 8000)
    emojis = ["👍", "🤔", "👎", "💀"]
    for i, thresh in enumerate(thresholds):
        if eval_diff < thresh: return emojis[i]
    return emojis[-1]

# ==========================================
# 4. СЕССИИ
# ==========================================
class HumanVsAISession:
    def __init__(self):
        self.agent = agent_absolute
        self.reset_game(first_player=1)

    def reset_game(self, first_player=1):
        self.my_board = self.opp_board = 0
        self.current_player = self.human_player = first_player
        self.agent_player = -first_player
        self.history = []
        self.game_over = False
        self.winning_line = 0
        self.evaluation_history = [0]
        self.hint_cell = -1
        return self.render_ui(), "Игра начата! Твой ход." if first_player == 1 else "Агент думает...", self.create_evaluation_chart()

    def save_state(self):
        self.history.append((self.my_board, self.opp_board, self.current_player, self.game_over, self.winning_line, self.evaluation_history.copy()))

    def undo(self):
        if len(self.history) >= 2 and not self.game_over:
            self.history.pop()
            prev = self.history.pop()
            self.my_board, self.opp_board, self.current_player, self.game_over, self.winning_line, self.evaluation_history = prev
            self.hint_cell = -1
            return self.render_ui(), "Ход отменен. Твой ход.", self.create_evaluation_chart()
        return self.render_ui(), "Нельзя отменить этот ход.", self.create_evaluation_chart()

    def create_evaluation_chart(self):
        fig = go.Figure()
        fig.add_trace(go.Scatter(y=self.evaluation_history, mode='lines+markers', name='Оценка', line=dict(color='#00ff00', width=2), marker=dict(size=8)))
        fig.update_layout(title='📊 Динамика оценки позиции', xaxis_title='Ход', yaxis_title='Оценка', template='plotly_dark', height=250, margin=dict(l=20, r=20, t=40, b=20))
        return fig

    def render_ui(self):
        values = []
        for i in range(49):
            bit = 1 << i
            is_win, is_hint = bool(self.winning_line & bit), (i == self.hint_cell)
            prefix = "🟩" if is_win else ("💡" if is_hint else "")
            variant = "primary" if (is_win or is_hint) else "secondary"
            if self.my_board & bit: values.append(gr.Button(value=f"{prefix}❌", variant=variant))
            elif self.opp_board & bit: values.append(gr.Button(value=f"{prefix}⭕", variant=variant))
            else: values.append(gr.Button(value=f"{prefix}·" if prefix else "·", variant=variant))
        return values

    def show_hint(self):
        if self.game_over or self.current_player != self.human_player:
            return self.render_ui(), "Сейчас нельзя показать подсказку.", self.create_evaluation_chart()
        move, _ = self.agent.find_best_move_and_eval(self.my_board, self.opp_board) if self.human_player == 1 else self.agent.find_best_move_and_eval(self.opp_board, self.my_board)
        self.hint_cell = move
        return self.render_ui(), f"💡 Рекомендую клетку {move}", self.create_evaluation_chart()

    def make_move(self, idx):
        if self.game_over or self.current_player != self.human_player:
            return self.render_ui(), "Сейчас не твой ход или игра окончена.", self.create_evaluation_chart()
        bit = 1 << idx
        if (self.my_board & bit) or (self.opp_board & bit):
            return self.render_ui(), "Клетка занята!", self.create_evaluation_chart()

        self.save_state()
        self.hint_cell = -1
        
        if self.human_player == 1:
            best_move, best_eval = self.agent.find_best_move_and_eval(self.my_board, self.opp_board)
            self.my_board |= bit
            actual_eval = self.agent._evaluate(self.my_board, self.opp_board)
        else:
            best_move, best_eval = self.agent.find_best_move_and_eval(self.opp_board, self.my_board)
            self.opp_board |= bit
            actual_eval = self.agent._evaluate(self.opp_board, self.my_board)
        
        move_quality = evaluate_move_quality(idx, actual_eval, best_move, best_eval, mode="win")
        win, w_line = bit_utils.check_win_and_line(self.my_board if self.human_player == 1 else self.opp_board)
        if win:
            self.game_over, self.winning_line = True, w_line
            self.evaluation_history.append(100000 if self.human_player == 1 else -100000)
            return self.render_ui(), f"🎉 Ты выиграл! {move_quality}", self.create_evaluation_chart()

        self.evaluation_history.append(actual_eval)
        self.current_player = self.agent_player
        start_time = time.time()
        
        if self.agent_player == 1:
            move = self.agent.get_move(self.my_board, self.opp_board)
            self.my_board |= (1 << move)
            win, w_line = bit_utils.check_win_and_line(self.my_board)
            eval_after = self.agent._evaluate(self.my_board, self.opp_board)
        else:
            move = self.agent.get_move(self.opp_board, self.my_board)
            self.opp_board |= (1 << move)
            win, w_line = bit_utils.check_win_and_line(self.opp_board)
            eval_after = self.agent._evaluate(self.opp_board, self.my_board)
            
        think_time = time.time() - start_time
        self.evaluation_history.append(eval_after)
        stats_text = f"🧠 Оценено: {self.agent.stats['positions_evaluated']}, Глубина: {self.agent.stats['max_depth_reached']}"
        
        if win:
            self.game_over, self.winning_line = True, w_line
            self.evaluation_history.append(100000 if self.agent_player == 1 else -100000)
            return self.render_ui(), f"😢 Агент выиграл! ({think_time:.3f}с)\n{stats_text}", self.create_evaluation_chart()
        
        self.current_player = self.human_player
        return self.render_ui(), f"Агент: {move} ({think_time:.3f}с). Твой ход.\n{stats_text}", self.create_evaluation_chart()

class SpectatorSession:
    def __init__(self):
        self.agents_dict = {
            "Absolute (Depth 6)": agent_absolute,
            "Grandmaster (Depth 8, 10s)": agent_grandmaster,
            "MCTS (1000 sim)": agent_mcts,
            "Strong Bitboard (Depth 4)": agent_strong,
            "Classic TFB (NumPy)": agent_classic,
            "🧬 Evolution Champion": agent_champion,  # ← НОВЫЙ БОЕЦ
        }
        self.reset_match()

    def reset_match(self, agent1_name="Absolute (Depth 6)", agent2_name="Grandmaster (Depth 8, 10s)"):
        self.agent1, self.agent2 = self.agents_dict[agent1_name], self.agents_dict[agent2_name]
        self.agent1_name, self.agent2_name = agent1_name, agent2_name
        self.my_board = self.opp_board = 0
        self.current_player = 1
        self.game_over = False
        self.winning_line = 0
        self.history = [f"--- Матч: {self.agent1_name} vs {self.agent2_name} ---"]
        self.move_count = 0
        self.evaluation_history = [0]
        self.total_stats = {"agent1_evals": 0, "agent2_evals": 0}
        return self.render_ui(), "\n".join(self.history), self.create_evaluation_chart(), self.create_stats_display()

    def create_evaluation_chart(self):
        fig = go.Figure()
        fig.add_trace(go.Scatter(y=self.evaluation_history, mode='lines+markers', name='Оценка', line=dict(color='#00ff00', width=2), marker=dict(size=6)))
        fig.update_layout(title='📊 Динамика оценки', xaxis_title='Полуход', yaxis_title='Оценка', template='plotly_dark', height=200, margin=dict(l=20, r=20, t=40, b=20))
        return fig

    def create_stats_display(self):
        return f"""<div style="background: #2d2d2d; padding: 15px; border-radius: 5px; font-family: monospace;">
            <h3 style="color: #00ff00; margin-top: 0;">🧠 Статистика мышления</h3>
            <p style="color: #fff; margin: 5px 0;"><strong>{self.agent1_name}:</strong> {self.total_stats['agent1_evals']} позиций</p>
            <p style="color: #fff; margin: 5px 0;"><strong>{self.agent2_name}:</strong> {self.total_stats['agent2_evals']} позиций</p>
            <p style="color: #888; margin: 5px 0; font-size: 12px;">Всего ходов: {self.move_count}</p>
        </div>"""

    def render_ui(self):
        values = []
        for i in range(49):
            bit = 1 << i
            is_win = bool(self.winning_line & bit)
            prefix = "🟩" if is_win else ""
            variant = "primary" if is_win else "secondary"
            if self.my_board & bit: values.append(gr.Button(value=f"{prefix}❌", variant=variant))
            elif self.opp_board & bit: values.append(gr.Button(value=f"{prefix}⭕", variant=variant))
            else: values.append(gr.Button(value="·", variant=variant))
        return values

    def step(self, steps=1):
        if self.game_over:
            return self.render_ui(), "\n".join(self.history) + "\n[Игра окончена]", self.create_evaluation_chart(), self.create_stats_display()
        
        for _ in range(steps):
            if self.game_over: break
            start_time = time.time()
            if self.current_player == 1:
                if "Classic" in self.agent1_name:
                    np_board = np.zeros(49, dtype=np.int8)
                    for j in range(49):
                        if self.my_board & (1 << j): np_board[j] = 1
                        elif self.opp_board & (1 << j): np_board[j] = -1
                    best_move, best_eval = self.agent1.find_best_move_and_eval(np_board, 1)
                    move = self.agent1.get_move(np_board, 1)
                    self.my_board |= (1 << move)
                    eval_after = self.agent1._evaluate(np_board, -np_board)
                else:
                    best_move, best_eval = self.agent1.find_best_move_and_eval(self.my_board, self.opp_board)
                    move = self.agent1.get_move(self.my_board, self.opp_board)
                    self.my_board |= (1 << move)
                    eval_after = self.agent1._evaluate(self.my_board, self.opp_board)
                win, w_line = bit_utils.check_win_and_line(self.my_board)
                player_name = "❌ (Крестики)"
                self.total_stats['agent1_evals'] += self.agent1.stats['positions_evaluated']
            else:
                if "Classic" in self.agent2_name:
                    np_board = np.zeros(49, dtype=np.int8)
                    for j in range(49):
                        if self.my_board & (1 << j): np_board[j] = 1
                        elif self.opp_board & (1 << j): np_board[j] = -1
                    best_move, best_eval = self.agent2.find_best_move_and_eval(np_board, -1)
                    move = self.agent2.get_move(np_board, -1)
                    self.opp_board |= (1 << move)
                    eval_after = self.agent2._evaluate(-np_board, np_board)
                else:
                    best_move, best_eval = self.agent2.find_best_move_and_eval(self.opp_board, self.my_board)
                    move = self.agent2.get_move(self.opp_board, self.my_board)
                    self.opp_board |= (1 << move)
                    eval_after = self.agent2._evaluate(self.opp_board, self.my_board)
                win, w_line = bit_utils.check_win_and_line(self.opp_board)
                player_name = "⭕ (Нолики)"
                self.total_stats['agent2_evals'] += self.agent2.stats['positions_evaluated']
                
            think_time = time.time() - start_time
            self.move_count += 1
            r, c = divmod(move, 7)
            move_quality = evaluate_move_quality(move, eval_after, best_move, best_eval, mode="win")
            self.history.append(f"Ход {self.move_count}: {player_name} -> {move} (р{r+1}, к{c+1}) [{think_time:.3f}с] {move_quality}")
            self.evaluation_history.append(eval_after)
            
            if win:
                self.game_over, self.winning_line = True, w_line
                self.history.append(f"🏆 ПОБЕДА: {player_name}!")
                self.evaluation_history.append(100000 if self.current_player == 1 else -100000)
                break
            self.current_player = -self.current_player
            if steps > 1: time.sleep(0.05)
        return self.render_ui(), "\n".join(self.history[-15:]), self.create_evaluation_chart(), self.create_stats_display()

class EvolutionSession:
    def __init__(self):
        self.population = []
        self.generation = 0
        self.history_scores = []
        self.history_weights = []
        self.best_agent_ever = None
        self.best_score_ever = -float('inf')
        self.last_ranked = None
        self.is_initialized = False
    
    def initialize(self, pop_size=12):
        self.population = []
        self.generation = 0
        self.history_scores = []
        self.history_weights = []
        self.best_agent_ever = None
        self.best_score_ever = -float('inf')
        
        # АГРЕССИВНЫЕ базовые веса
        base_weights = {
            'w2': 50, 'w3': 15000, 'w_open_three': 8000,
            'w_fork': 12000, 'w_center': 100, 'block_penalty': 6000,
        }
        
        for _ in range(pop_size):
            weights = {k: v * (0.5 + random.random()) for k, v in base_weights.items()}
            self.population.append(EvolvableAgent(
                weights=weights, depth=3, time_limit=0.05, temperature=0.25
            ))
        
        self.is_initialized = True
        return f"✅ Популяция создана: {pop_size} агентов (Режим: АГРЕССИЯ + ХАОС 25%)"
    
    def run_generation(self, games_per_pair=2):
        if not self.is_initialized:
            return "❌ Сначала создайте популяцию!", None, None, None
        
        self.generation += 1
        gen_start = time.time()
        scores = [0] * len(self.population)
        for i in range(len(self.population)):
            for j in range(i+1, len(self.population)):
                for _ in range(games_per_pair // 2):
                    # Упрощенный турнир для скорости
                    r1 = self._play_game(self.population[i], self.population[j])
                    scores[i] += (1 if r1 == 1 else 0) + (0.5 if r1 == 0 else 0)
                    scores[j] += (1 if r1 == -1 else 0) + (0.5 if r1 == 0 else 0)
                    r2 = self._play_game(self.population[j], self.population[i])
                    scores[j] += (1 if r2 == 1 else 0) + (0.5 if r2 == 0 else 0)
                    scores[i] += (1 if r2 == -1 else 0) + (0.5 if r2 == 0 else 0)
        
        ranked = sorted(enumerate(scores), key=lambda x: -x[1])
        self.last_ranked = [(idx, score, self.population[idx]) for idx, score in ranked]
        top_score, avg_score, worst_score = ranked[0][1], sum(s for _, s in ranked) / len(ranked), ranked[-1][1]
        gen_time = time.time() - gen_start
        self.history_scores.append((self.generation, top_score, avg_score, worst_score))
        
        best_agent = self.population[ranked[0][0]]
        self.history_weights.append({'gen': self.generation, **best_agent.weights.copy()})
        if top_score > self.best_score_ever:
            self.best_score_ever = top_score
            self.best_agent_ever = best_agent.clone()
        
        new_population = [self.population[ranked[i][0]].clone() for i in range(min(3, len(self.population)))]
        while len(new_population) < len(self.population):
            pool = random.sample(range(len(self.population)), min(4, len(self.population)))
            pool_ranked = sorted(pool, key=lambda i: -scores[i])
            child = self.population[pool_ranked[0]].crossover(self.population[pool_ranked[1]])
            child.mutate(mutation_rate=0.4, mutation_scale=0.3)
            new_population.append(child)
        self.population = new_population
        
        log_text = f"📊 Поколение {self.generation}\n🏆 Лучший: {top_score:.1f} | 📈 Средний: {avg_score:.1f}\n⏱️ Время: {gen_time:.1f}с | 🎯 Рекорд: {self.best_score_ever:.1f}"
        return log_text, self._create_progress_chart(), self._create_weights_chart(), self._create_leaderboard()

    def _play_game(self, agent1, agent2):
        my1, opp1, my2, opp2, current, moves = 0, 0, 0, 0, 1, 0
        while moves < 49:
            if current == 1:
                move = agent1.get_move(my1, opp1)
                my1 |= (1 << move); opp2 |= (1 << move)
                if bit_utils.check_win_and_line(my1)[0]: return 1
            else:
                move = agent2.get_move(my2, opp2)
                my2 |= (1 << move); opp1 |= (1 << move)
                if bit_utils.check_win_and_line(my2)[0]: return -1
            current, moves = -current, moves + 1
        return 0
    
    def _create_progress_chart(self):
        if not self.history_scores: return None
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[h[0] for h in self.history_scores], y=[h[1] for h in self.history_scores], mode='lines+markers', name='🏆 Лучший', line=dict(color='gold', width=3)))
        fig.add_trace(go.Scatter(x=[h[0] for h in self.history_scores], y=[h[2] for h in self.history_scores], mode='lines+markers', name='📈 Средний', line=dict(color='#00ff00', width=2)))
        fig.update_layout(title='📊 Прогресс эволюции', xaxis_title='Поколение', yaxis_title='Очки', template='plotly_dark', height=300, margin=dict(l=20, r=20, t=40, b=20))
        return fig
    
    def _create_weights_chart(self):
        if not self.history_weights: return None
        fig = px.line(pd.DataFrame(self.history_weights), x='gen', y=EvolvableAgent.PARAM_NAMES, title='🧬 Эволюция весов', template='plotly_dark', height=300)
        fig.update_layout(xaxis_title='Поколение', yaxis_title='Значение')
        return fig
    
    def _create_leaderboard(self):
        if not self.last_ranked: return "Турнир ещё не проводился"
        text = "🏆 ТОП-5 АГЕНТОВ\n" + "="*40 + "\n"
        for rank, (idx, score, agent) in enumerate(self.last_ranked[:5], 1):
            text += f"{['🥇','🥈','🥉','4️⃣','5️⃣'][rank-1]} #{rank}: {score:.1f} очков\n   Веса: " + ", ".join([f"{k}={v:.0f}" for k, v in agent.weights.items()]) + "\n\n"
        return text
    
    def play_best_game(self):
        if not self.best_agent_ever:
            return "⚠️ Чемпион еще не определен. Проведите хотя бы 1 поколение.", "<b>Доска пуста</b>"
        
        baseline = EvolvableAgent(depth=3, time_limit=0.05) # Базовый агент с теми же настройками
        my1, opp1, my2, opp2, current, moves = 0, 0, 0, 0, 1, 0
        history = ["🎮 Партия: Эволюционный чемпион vs Базовый агент\n"]
        
        while moves < 49:
            if current == 1:
                move = self.best_agent_ever.get_move(my1, opp1)
                my1 |= (1 << move); opp2 |= (1 << move)
                r, c = divmod(move, 7)
                history.append(f"Ход {moves+1}: ❌ Чемпион -> клетка {move} (р{r+1}, к{c+1})")
                if bit_utils.check_win_and_line(my1)[0]:
                    history.append("\n🏆 ПОБЕДА: Эволюционный чемпион!"); break
            else:
                move = baseline.get_move(my2, opp2)
                my2 |= (1 << move); opp1 |= (1 << move)
                r, c = divmod(move, 7)
                history.append(f"Ход {moves+1}: ⭕ Базовый -> клетка {move} (р{r+1}, к{c+1})")
                if bit_utils.check_win_and_line(my2)[0]:
                    history.append("\n🏆 ПОБЕДА: Базовый агент!"); break
            current, moves = -current, moves + 1
        else:
            history.append("\n🤝 Ничья")
        
        board_html = "<table style='border-collapse: collapse; margin: auto;'>"
        for r in range(7):
            board_html += "<tr>"
            for c in range(7):
                idx = r * 7 + c
                bit = 1 << idx
                symbol = "❌" if my1 & bit else ("⭕" if opp1 & bit else "·")
                board_html += f"<td style='width:40px; height:40px; text-align:center; border:1px solid #555; font-size:20px;'>{symbol}</td>"
            board_html += "</tr>"
        board_html += "</table>"
        return "\n".join(history), board_html
    
    def get_best_weights_text(self):
        if not self.best_agent_ever:
            return "⚠️ Чемпион еще не определен. Проведите хотя бы 1 поколение."
        return f"🏆 Веса лучшего агента (рекорд: {self.best_score_ever:.1f} очков):\n\n{self.best_agent_ever.weights}"

human_session = HumanVsAISession()
spectator_session = SpectatorSession()
evo_session = EvolutionSession()

# ==========================================
# 5. GRADIO ИНТЕРФЕЙС
# ==========================================
def create_evolution_tab():
    gr.Markdown("# 🧬 Лаборатория эволюции\nНаблюдайте, как агенты учатся играть через естественный отбор.")
    with gr.Row():
        with gr.Column(scale=1):
            pop_slider = gr.Slider(8, 20, value=12, step=2, label="👥 Размер популяции")
            games_slider = gr.Slider(2, 4, value=2, step=2, label="🎮 Игр на пару")
            init_btn = gr.Button("🌱 Создать популяцию", variant="primary")
            step_btn = gr.Button("⏭️ Следующее поколение", variant="secondary")
            auto_btn = gr.Button("🔄 Запустить 10 поколений", variant="secondary")
            gr.Markdown("### 🎮 Анализ лучшего агента")
            play_btn = gr.Button("▶️ Показать партию чемпиона", variant="secondary")
            weights_btn = gr.Button("📋 Получить веса чемпиона", variant="secondary")
        with gr.Column(scale=2):
            log_box = gr.Textbox(label="📜 Лог эволюции", lines=6, interactive=False)
            progress_chart = gr.Plot(label="📊 Прогресс")
            weights_chart = gr.Plot(label="🧬 Эволюция весов")
            leaderboard = gr.Textbox(label="🏆 Таблица лидеров", lines=8, interactive=False)
            with gr.Accordion("🎮 Партия чемпиона", open=False):
                game_log = gr.Textbox(label="Ходы партии", lines=8, interactive=False)
                game_board = gr.HTML(label="Доска")
            weights_output = gr.Textbox(label="💾 Веса для копирования", lines=4, interactive=False)

    # ОБРАБОТЧИКИ
    def init_handler(pop_size):
        return evo_session.initialize(int(pop_size))
    
    def step_handler(games):
        return evo_session.run_generation(int(games))
    
    # ИСПРАВЛЕНИЕ 1: Запуск 10 поколений в цикле
    def auto_handler(games):
        result = None
        for i in range(10):
            result = evo_session.run_generation(int(games))
        return result
    
    # ИСПРАВЛЕНИЕ 2 и 3: Явные обёртки для партии и весов
    def play_handler():
        log_text, board_html = evo_session.play_best_game()
        return log_text, board_html
    
    def weights_handler():
        weights_text = evo_session.get_best_weights_text()
        return weights_text

    # Привязка обработчиков
    init_btn.click(fn=init_handler, inputs=[pop_slider], outputs=[log_box])
    step_btn.click(fn=step_handler, inputs=[games_slider], outputs=[log_box, progress_chart, weights_chart, leaderboard])
    auto_btn.click(fn=auto_handler, inputs=[games_slider], outputs=[log_box, progress_chart, weights_chart, leaderboard])
    play_btn.click(fn=play_handler, inputs=[], outputs=[game_log, game_board])
    weights_btn.click(fn=weights_handler, inputs=[], outputs=[weights_output])

def create_gradio_interface():
    css_code = """
    .agent-vs-agent-log { font-family: monospace; font-size: 13px; background: #1e1e1e; color: #00ff00; padding: 10px; border-radius: 5px; height: 300px; overflow-y: auto; }
    button { transition: all 0.3s ease; }
    button:hover { transform: scale(1.05); }
    """
    with gr.Blocks(title="Ultimate TicTacToe 7x7") as demo:
        gr.Markdown("# 🏆 Ultimate Крестики-нолики 7x7 (4 в ряд)")
        gr.Markdown("**v3.0**: Детектор открытых триплетов + Оценка ходов эмодзи + 🧬 Лаборатория эволюции")
        
        with gr.Tabs():
            with gr.TabItem("⚔️ Играть против ИИ"):
                with gr.Row():
                    fp_radio = gr.Radio(choices=[("Ты (❌)", 1), ("Агент (⭕)", -1)], label="Кто ходит первым?", value=1)
                    reset_btn = gr.Button("🔄 Новая игра", variant="primary")
                    undo_btn = gr.Button("↩️ Отменить ход", variant="secondary")
                    hint_btn = gr.Button("💡 Подсказка", variant="secondary")
                status = gr.Textbox(value="Нажми 'Новая игра' для начала", label="Статус", interactive=False)
                grid = [[gr.Button("·", size="sm", min_width=50, elem_id=f"btn_{i*7+j}") for j in range(7)] for i in range(7)]
                all_btns = [btn for row in grid for btn in row]
                eval_chart = gr.Plot(label="📊 График оценки")
                
                for i in range(7):
                    for j in range(7):
                        idx = i * 7 + j
                        grid[i][j].click(fn=lambda i=idx: human_session.make_move(i), inputs=[], outputs=all_btns + [status, eval_chart])
                reset_btn.click(lambda fp: human_session.reset_game(first_player=fp), inputs=[fp_radio], outputs=all_btns + [status, eval_chart])
                undo_btn.click(human_session.undo, inputs=[], outputs=all_btns + [status, eval_chart])
                hint_btn.click(human_session.show_hint, inputs=[], outputs=all_btns + [status, eval_chart])

            with gr.TabItem("👁️ Режим наблюдателя (AI vs AI)"):
                gr.Markdown("**Система оценки:** 🌟 Идеальный | 👍 Хороший | 🤔 Сомнительный | 👎 Плохой | 💀 Грубая ошибка")
                with gr.Row():
                    with gr.Column(scale=1):
                        agent1_drop = gr.Dropdown(choices=list(spectator_session.agents_dict.keys()), value="Absolute (Depth 6)", label="Игрок 1 (❌)")
                        agent2_drop = gr.Dropdown(choices=list(spectator_session.agents_dict.keys()), value="Grandmaster (Depth 8, 10s)", label="Игрок 2 (⭕)")
                        spec_reset_btn = gr.Button("🔄 Начать матч", variant="primary")
                        with gr.Row():
                            step1_btn = gr.Button("▶ 1 ход", variant="secondary")
                            step3_btn = gr.Button("⏩ 3 хода", variant="secondary")
                            step10_btn = gr.Button("⏭️ 10 ходов", variant="secondary")
                    with gr.Column(scale=2):
                        spec_status = gr.Textbox(label="Журнал ходов", lines=10, interactive=False, elem_classes=["agent-vs-agent-log"])
                        stats_display = gr.HTML(value="<div>Нажми 'Начать матч'</div>")
                spec_grid = [[gr.Button("·", size="sm", min_width=50) for j in range(7)] for i in range(7)]
                spec_all_btns = [btn for row in spec_grid for btn in row]
                spec_eval_chart = gr.Plot(label="📊 Динамика оценки")
                
                spec_reset_btn.click(lambda a1, a2: spectator_session.reset_match(a1, a2), inputs=[agent1_drop, agent2_drop], outputs=spec_all_btns + [spec_status, spec_eval_chart, stats_display])
                step1_btn.click(lambda: spectator_session.step(1), inputs=[], outputs=spec_all_btns + [spec_status, spec_eval_chart, stats_display])
                step3_btn.click(lambda: spectator_session.step(3), inputs=[], outputs=spec_all_btns + [spec_status, spec_eval_chart, stats_display])
                step10_btn.click(lambda: spectator_session.step(10), inputs=[], outputs=spec_all_btns + [spec_status, spec_eval_chart, stats_display])
            
            with gr.TabItem("🧬 Эволюция"):
                create_evolution_tab()

    return demo, css_code

if __name__ == "__main__":
    demo, css_code = create_gradio_interface()
    print("🚀 Запуск Ultimate TicTacToe 7x7 v3.0...")
    print("✨ Включено: Детектор открытых триплетов, Оценка ходов эмодзи, Лаборатория эволюции")
    demo.launch(css=css_code)

🚀 Запуск Ultimate TicTacToe 7x7 v3.0...
✨ Включено: Детектор открытых триплетов, Оценка ходов эмодзи, Лаборатория эволюции
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
